In [2]:
import os
import pandas as pd
import pandas_gbq
import numpy as np
from pandas.api.types import is_numeric_dtype
from google.cloud import bigquery

CREDS = '../../../converge-database-0331482f2ee5.json'
PROJECT_ID = "converge-database"
client = bigquery.Client.from_service_account_json(json_credentials_path=CREDS)

# =========================================================

SOURCE_SERIATIM  = "converge-database.aclico.seriatim_values"
SOURCE_POLICY    = "converge-database.aclico.policy"
SOURCE_TERM      = "converge-database.aclico.term_table"
SOURCE_SURRENDER = "converge-database.aclico.surrender_fees"
DEST_TABLE       = "converge-database.All_Contracts_MYGA.All_Contracts_ACLICO"
STAGING_TABLE    = f"{PROJECT_ID}.scratch._stg_all_contracts_aclico_fixed_rebuild"

MERGE_MODE = "sync"

OUT_MISMATCH_XLSX = "Aclico_historical_mismatch.xlsx"
OUT_TARGET_DIFF   = "Aclico_target_vs_rebuild_diff.csv"

# =========================================================
# 1) Helpers
# =========================================================
def unify_series(series: pd.Series):
    """
    For a given field under the same policy_number, return (value, is_mismatch):

    - All NA -> (None, False)
    - Numeric and all 0 -> (0, False)
    - If only +/- sign differs -> (abs(x), False)
    - If exactly one unique non-zero -> (that value, False)
    - Else multiple different values -> (None, True)
    """
    clean = series.dropna()
    if clean.empty:
        return None, False

    if is_numeric_dtype(series):
        if (clean == 0).all():
            return 0, False
        vals = clean[clean != 0].tolist()
    else:
        vals = clean.tolist()

    unique_vals = set(vals)

    # +/- same abs -> abs
    if is_numeric_dtype(series) and len(unique_vals) > 1:
        abs_set = {abs(v) for v in unique_vals}
        if len(abs_set) == 1:
            return abs_set.pop(), False

    if len(unique_vals) == 1:
        return unique_vals.pop(), False

    return None, True

def first_nonnull(s: pd.Series):
    s2 = s.dropna()
    return s2.iloc[0] if len(s2) else None

def detect_transitions(g_sorted: pd.DataFrame, col: str) -> list:
    records = []
    prev_val = None
    for _, r in g_sorted[["set_month", col]].iterrows():
        cur_val = r[col]
        cur_month = r["set_month"]
        if pd.isna(cur_val):
            continue
        if prev_val is not None:
            same = str(cur_val) == str(prev_val)
            if not same and is_numeric_dtype(g_sorted[col]):
                try:
                    same = abs(float(cur_val)) == abs(float(prev_val))
                except Exception:
                    pass
            if not same:
                records.append({
                    "changed_at_set_month": cur_month,
                    "from_value": prev_val,
                    "to_value": cur_val,
                })
        prev_val = cur_val
    return records

# =========================================================
# 2) Auto cutoff month (seriatim only — no premium table for Aclico)
# =========================================================
q_max = f"SELECT MAX(set_month) AS max_ser FROM `{SOURCE_SERIATIM}`"
mx = client.query(q_max).to_dataframe().iloc[0]
if pd.isna(mx["max_ser"]):
    raise ValueError("Cannot determine max set_month from seriatim table.")
CUTOFF_SET_MONTH = str(mx["max_ser"])
print(f"[INFO] Using cutoff set_month <= {CUTOFF_SET_MONTH}")

# =========================================================
# 3) Read ALL historical seriatim up to cutoff
# =========================================================
query_ser = f"""
SELECT 
  policy_number,
  plan,
  deposit_type        AS myga_dep_type,
  reins_flag,
  date_approved,
  tax_qualified,
  purchase_price,
  rider_1             AS death_benefit_rider,
  rider_2             AS accu_inter_withdrawal_rider,
  rider_3             AS free_partial_withdrawal_rider,
  reins_pct           AS quota_share,
  set_month
FROM `{SOURCE_SERIATIM}`
WHERE set_month <= '{CUTOFF_SET_MONTH}'
"""
df_ser = pandas_gbq.read_gbq(query_ser, project_id=PROJECT_ID, credentials=client._credentials)
df_ser["policy_number"] = df_ser["policy_number"].astype(str)

FIXED_COLS_SER = [
    "plan", "myga_dep_type", "reins_flag", "date_approved", "tax_qualified",
    "purchase_price", "death_benefit_rider", "accu_inter_withdrawal_rider",
    "free_partial_withdrawal_rider", "quota_share"
]

# =========================================================
# 4) Historical fixed snapshot from seriatim + mismatch list
# =========================================================
rows = []
mismatch_records = []

for pol, g in df_ser.groupby("policy_number", dropna=False):
    pol = str(pol)
    row = {"policy_number": pol}
    g_sorted = g.sort_values("set_month")
    for col in FIXED_COLS_SER:
        val, mm = unify_series(g[col])
        row[col] = val
        if mm:
            for t in detect_transitions(g_sorted, col):
                mismatch_records.append({"policy_number": pol, "variable": col, **t})
    rows.append(row)

fixed_ser = pd.DataFrame(rows)

for c in ["date_approved"]:
    if c in fixed_ser.columns:
        fixed_ser[c] = pd.to_datetime(fixed_ser[c], errors="coerce").dt.date

# =========================================================
# =========================================================
# 5) Read policy table (static — no set_month)
# =========================================================
query_policy = f"""
SELECT
  CAST(policy_number AS STRING) AS policy_number,
  DATE(date_issued)              AS issue_date,
  issue_state,
  issue_age
FROM `{SOURCE_POLICY}`
"""
df_policy = pandas_gbq.read_gbq(query_policy, project_id=PROJECT_ID, credentials=client._credentials)
df_policy["policy_number"] = df_policy["policy_number"].astype(str)
df_policy["issue_date"] = pd.to_datetime(df_policy["issue_date"], errors="coerce").dt.date
df_policy["issue_age"] = pd.to_numeric(df_policy["issue_age"], errors="coerce").astype("Int64")

# Deduplicate — keep first row per policy
df_policy_fixed = df_policy.drop_duplicates("policy_number", keep="first")[
    ["policy_number", "issue_date", "issue_state", "issue_age"]
]

# =========================================================
# 6) Read term_table (static — one row per plan, joined on plan)
# =========================================================
query_term_tbl = f"""
SELECT gender, term, plan
FROM `{SOURCE_TERM}`
"""
df_term_tbl = pandas_gbq.read_gbq(query_term_tbl, project_id=PROJECT_ID, credentials=client._credentials)

# =========================================================
# 7) Surrender from surrender_fees table
# =========================================================
query_surrender = f"""
SELECT
  CAST(policy_number AS STRING) AS policy_number,
  DATE(date)                     AS surrender_date
FROM `{SOURCE_SURRENDER}`
"""
try:
    df_surrender = pandas_gbq.read_gbq(query_surrender, project_id=PROJECT_ID, credentials=client._credentials)
except Exception as e:
    print(f"[WARN] surrender_fees table read failed.\n{e}")
    df_surrender = pd.DataFrame(columns=["policy_number", "surrender_date"])

df_surrender["policy_number"] = df_surrender["policy_number"].astype(str)
df_surrender["surrender_date"] = pd.to_datetime(df_surrender["surrender_date"], errors="coerce").dt.date

fixed_surrender = (
    df_surrender.sort_values("surrender_date")
               .drop_duplicates("policy_number", keep="first")
               [["policy_number", "surrender_date"]]
)

# =========================================================
# 8) Final rebuilt fixed table
# =========================================================
df_final = (
    fixed_ser
    .merge(df_policy_fixed, on="policy_number", how="left")
    .merge(df_term_tbl,     on="plan",           how="left")
    .merge(fixed_surrender, on="policy_number",  how="left")
)

final_columns = [
    "policy_number",
    "plan",
    "reins_flag",
    "myga_dep_type",
    "issue_date",
    "date_approved",
    "issue_state",
    "issue_age",
    "gender",
    "term",
    "tax_qualified",
    "purchase_price",
    "death_benefit_rider",
    "free_partial_withdrawal_rider",
    "accu_inter_withdrawal_rider",
    "quota_share",
    "surrender_date",
]
df_final = df_final.reindex(columns=final_columns)

for c in ["issue_age", "term"]:
    if c in df_final.columns:
        df_final[c] = pd.to_numeric(df_final[c], errors="coerce")
        bad = df_final[c].notna() & (df_final[c] % 1 != 0)
        if bad.any():
            raise ValueError(f"Non-integer values found in {c} after merge: {df_final.loc[bad, c].head().tolist()}")
        df_final[c] = df_final[c].astype("Int64")

# =========================================================
# 9) Save mismatch report
# =========================================================
pd.DataFrame(mismatch_records).drop_duplicates().to_excel(OUT_MISMATCH_XLSX, index=False)
print(f"[OK] Historical mismatch report: {OUT_MISMATCH_XLSX}")

# =========================================================
# 10) Diff report vs destination (field-level)
# =========================================================
query_dest = f"SELECT {', '.join(final_columns)} FROM `{DEST_TABLE}`"
df_dest = pandas_gbq.read_gbq(query_dest, project_id=PROJECT_ID, credentials=client._credentials)
df_dest["policy_number"] = df_dest["policy_number"].astype(str)

comp = df_final.merge(df_dest, on="policy_number", how="inner", suffixes=("_src","_dest"))

diff_rows = []
for col in [c for c in final_columns if c != "policy_number"]:
    a = comp[f"{col}_src"]
    b = comp[f"{col}_dest"]
    both_null = a.isna() & b.isna()
    neq = (~both_null) & (a.astype(str) != b.astype(str))
    if neq.any():
        tmp = comp.loc[neq, ["policy_number", f"{col}_src", f"{col}_dest"]].copy()
        tmp.insert(1, "variable", col)
        tmp = tmp.rename(columns={f"{col}_src": "rebuilt_value", f"{col}_dest": "dest_value"})
        diff_rows.append(tmp)

df_diff = pd.concat(diff_rows, ignore_index=True) if diff_rows else pd.DataFrame(
    columns=["policy_number", "variable", "rebuilt_value", "dest_value"]
)
df_diff.to_csv(OUT_TARGET_DIFF, index=False)
print(f"[OK] Target vs rebuilt diff report: {OUT_TARGET_DIFF}")

# =========================================================
# 11) Load staging
# =========================================================
job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
client.load_table_from_dataframe(df_final, STAGING_TABLE, job_config=job_config).result()
print(f"[OK] Staging loaded: {STAGING_TABLE}")

# =========================================================
# 12) MERGE with SAFE_CAST
# =========================================================
dest_table_obj = client.get_table(DEST_TABLE)
dest_type_map = {f.name: f.field_type.upper() for f in dest_table_obj.schema}

def normalize_bq_type(t: str) -> str:
    # BigQuery may return INTEGER/FLOAT/BOOLEAN instead of INT64/FLOAT64/BOOL
    return {
        "INTEGER": "INT64",
        "FLOAT":   "FLOAT64",
        "BOOLEAN": "BOOL",
    }.get(t, t)

def s_expr(col: str) -> str:
    if col == "policy_number":
        return "CAST(S.policy_number AS STRING)"
    t = dest_type_map.get(col)
    if t is None:
        return f"S.{col}"
    t = normalize_bq_type(t)
    if t in ("STRING","BYTES","INT64","FLOAT64","NUMERIC","BIGNUMERIC","BOOL",
             "DATE","DATETIME","TIMESTAMP","TIME"):
        return f"SAFE_CAST(S.{col} AS {t})"
    return f"S.{col}"

if MERGE_MODE.lower() != "sync":
    raise ValueError("This rebuild version expects MERGE_MODE='sync' so mismatches become NULL in destination.")

set_expr = ",\n      ".join([f"{c} = {s_expr(c)}" for c in final_columns if c != "policy_number"])

insert_cols_sql = ", ".join(final_columns)
insert_vals_sql = ", ".join([s_expr(c) for c in final_columns])

merge_sql = f"""
MERGE `{DEST_TABLE}` T
USING `{STAGING_TABLE}` S
ON T.policy_number = CAST(S.policy_number AS STRING)

WHEN MATCHED THEN
  UPDATE SET
      {set_expr}

WHEN NOT MATCHED THEN
  INSERT ({insert_cols_sql})
  VALUES ({insert_vals_sql})
"""
client.query(merge_sql).result()
print(f"[OK] MERGE completed into {DEST_TABLE} (mode={MERGE_MODE}).")

[INFO] Using cutoff set_month <= 202605
Downloading:  70%|██████▉   |

KeyboardInterrupt: 

In [ ]:
# Confirmed with SC, 202603 ACL MYGA issue age discrepancy is a cedent's admin system correction, informed by ACL on a previous call.